# DepegScope: Network Analysis

This notebook analyzes the stablecoin-protocol dependency network.

## Contents
1. Build Dependency Graph
2. Network Statistics
3. Centrality Analysis
4. Systemic Risk Identification
5. Visualization

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from src.models.stablecoin import Stablecoin
from src.models.protocol import Protocol
from src.models.exposure import Exposure
from src.analysis.dependency_graph import DependencyGraph
from src.visualization.network_plots import NetworkVisualizer
from config.settings import PROCESSED_DATA_DIR

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

%matplotlib inline

## 1. Build Dependency Graph

In [ ]:
# Load processed data
def load_data():
    stablecoins = []
    protocols = []
    exposures = []
    
    stables_file = PROCESSED_DATA_DIR / "stablecoins_processed.json"
    if stables_file.exists():
        with open(stables_file) as f:
            data = json.load(f)
        stablecoins = [Stablecoin.from_dict(s) for s in data]
    
    protocols_file = PROCESSED_DATA_DIR / "protocols_processed.json"
    if protocols_file.exists():
        with open(protocols_file) as f:
            data = json.load(f)
        protocols = [Protocol.from_dict(p) for p in data]
    
    exposures_file = PROCESSED_DATA_DIR / "exposures_processed.json"
    if exposures_file.exists():
        with open(exposures_file) as f:
            data = json.load(f)
        exposures = [Exposure.from_dict(e) for e in data]
    
    return stablecoins, protocols, exposures

stablecoins, protocols, exposures = load_data()
print(f"Loaded: {len(stablecoins)} stablecoins, {len(protocols)} protocols, {len(exposures)} exposures")

In [ ]:
# Build the dependency graph
graph = DependencyGraph()
graph.build_from_data(stablecoins, protocols, exposures)

summary = graph.summary()
print("Graph Summary:")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"  {key}: ${value:,.0f}" if 'usd' in key else f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

## 2. Network Statistics

In [ ]:
# Basic network statistics
G = graph.graph

print("=== Network Statistics ===")
print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density: {nx.density(G):.4f}")

# For undirected version
G_undirected = G.to_undirected()
if nx.is_connected(G_undirected):
    print(f"Average shortest path length: {nx.average_shortest_path_length(G_undirected):.2f}")
    print(f"Diameter: {nx.diameter(G_undirected)}")
else:
    print(f"Graph is not connected. Number of components: {nx.number_connected_components(G_undirected)}")

print(f"Average clustering coefficient: {nx.average_clustering(G_undirected):.4f}")

In [ ]:
# Degree distribution
degrees = [d for n, d in G.degree()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(degrees, bins=30, edgecolor='black')
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Degree Distribution')

# Log-log plot (power law check)
degree_counts = pd.Series(degrees).value_counts().sort_index()
axes[1].scatter(degree_counts.index, degree_counts.values)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Degree (log)')
axes[1].set_ylabel('Frequency (log)')
axes[1].set_title('Degree Distribution (Log-Log)')

plt.tight_layout()
plt.show()

## 3. Centrality Analysis

In [ ]:
# Calculate centrality metrics
centrality_df = graph.calculate_centrality_metrics()
display(centrality_df.head(20))

In [ ]:
# Top entities by different centrality measures
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

metrics = ['degree_centrality', 'betweenness_centrality', 'pagerank', 'eigenvector_centrality']
titles = ['Degree Centrality', 'Betweenness Centrality', 'PageRank', 'Eigenvector Centrality']

for ax, metric, title in zip(axes.flat, metrics, titles):
    if metric in centrality_df.columns:
        top_10 = centrality_df.nlargest(10, metric)
        colors = ['#e74c3c' if t == 'stablecoin' else '#3498db' for t in top_10['type']]
        ax.barh(top_10['entity'], top_10[metric], color=colors)
        ax.set_xlabel(metric.replace('_', ' ').title())
        ax.set_title(f'Top 10 by {title}')
        ax.invert_yaxis()

plt.tight_layout()
plt.show()

# Legend
print("Legend: Red = Stablecoin, Blue = Protocol")

In [ ]:
# Centrality comparison: stablecoins vs protocols
stablecoin_centrality = centrality_df[centrality_df['type'] == 'stablecoin']
protocol_centrality = centrality_df[centrality_df['type'] == 'protocol']

print("=== Centrality Comparison ===")
for metric in ['degree_centrality', 'betweenness_centrality', 'pagerank']:
    if metric in centrality_df.columns:
        print(f"\n{metric.replace('_', ' ').title()}:")
        print(f"  Stablecoins - Mean: {stablecoin_centrality[metric].mean():.4f}, Max: {stablecoin_centrality[metric].max():.4f}")
        print(f"  Protocols   - Mean: {protocol_centrality[metric].mean():.4f}, Max: {protocol_centrality[metric].max():.4f}")

## 4. Systemic Risk Identification

In [ ]:
# Find systemic chokepoints
chokepoints = graph.find_systemic_chokepoints()

print("=== Systemic Chokepoints (Stablecoins) ===")
print("(Ranked by potential TVL at risk from depeg)\n")

for i, (stablecoin, risk_score) in enumerate(chokepoints[:10], 1):
    print(f"{i:2d}. {stablecoin:10s} - ${risk_score:>15,.0f} at risk")

In [ ]:
# Blast radius for top stablecoins
top_stables = [s for s, _ in chokepoints[:5]]

fig, ax = plt.subplots(figsize=(10, 6))

severities = [0.05, 0.10, 0.20, 0.50]
x = np.arange(len(top_stables))
width = 0.2

for i, severity in enumerate(severities):
    blast_radii = [graph.calculate_blast_radius(s, severity) for s in top_stables]
    ax.bar(x + i*width, [b/1e9 for b in blast_radii], width, label=f'{severity*100:.0f}% depeg')

ax.set_xlabel('Stablecoin')
ax.set_ylabel('Blast Radius (Billions USD)')
ax.set_title('Blast Radius by Depeg Severity')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(top_stables)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Protocol vulnerability analysis
print("=== Most Vulnerable Protocols ===")
print("(Protocols with highest concentration in single stablecoin)\n")

# Calculate concentration for each protocol
vulnerability = []
for protocol in protocols:
    if protocol.stablecoin_holdings:
        total = sum(protocol.stablecoin_holdings.values())
        if total > 0:
            max_exposure = max(protocol.stablecoin_holdings.values())
            dominant = max(protocol.stablecoin_holdings.items(), key=lambda x: x[1])
            concentration = max_exposure / total
            vulnerability.append({
                'protocol': protocol.name,
                'concentration': concentration,
                'dominant_stablecoin': dominant[0],
                'tvl': protocol.tvl
            })

vuln_df = pd.DataFrame(vulnerability)
if not vuln_df.empty:
    vuln_df = vuln_df.sort_values('concentration', ascending=False)
    display(vuln_df.head(15))

## 5. Network Visualization

In [ ]:
# Simple network visualization
try:
    viz = NetworkVisualizer(graph)
    fig = viz.plot_dependency_network(layout='spring')
    plt.show()
except Exception as e:
    print(f"Visualization error: {e}")
    
    # Fallback: basic NetworkX visualization
    plt.figure(figsize=(15, 15))
    
    # Color by type
    colors = ['#e74c3c' if G.nodes[n].get('type') == 'stablecoin' else '#3498db' for n in G.nodes()]
    
    # Size by degree
    degrees = dict(G.degree())
    sizes = [100 + degrees[n] * 50 for n in G.nodes()]
    
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    nx.draw(G, pos, node_color=colors, node_size=sizes, alpha=0.7, 
            with_labels=False, edge_color='gray', edge_alpha=0.3)
    
    # Label high-degree nodes
    high_degree = {n: n for n, d in G.degree() if d > 10}
    nx.draw_networkx_labels(G, pos, labels=high_degree, font_size=8)
    
    plt.title('Stablecoin-Protocol Dependency Network')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# Subgraph: Focus on top stablecoin
if chokepoints:
    top_stable = chokepoints[0][0]
    
    # Get neighbors of top stablecoin
    neighbors = list(G.neighbors(top_stable)) + [top_stable]
    subgraph = G.subgraph(neighbors)
    
    plt.figure(figsize=(12, 12))
    
    colors = ['#e74c3c' if n == top_stable else '#3498db' for n in subgraph.nodes()]
    sizes = [1000 if n == top_stable else 300 for n in subgraph.nodes()]
    
    pos = nx.spring_layout(subgraph, k=1.5, seed=42)
    nx.draw(subgraph, pos, node_color=colors, node_size=sizes, alpha=0.8,
            with_labels=True, font_size=8, edge_color='gray')
    
    plt.title(f'Protocols Exposed to {top_stable}')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## Summary

Key findings from network analysis:
1. Network structure and density
2. Most central entities (stablecoins and protocols)
3. Systemic chokepoints and their blast radius
4. Vulnerable protocols with high concentration

## Next Steps

Continue to simulation scenarios:
- `03_simulation_scenarios.ipynb`